# Advanced Problems with Solutions: Decorators That Prime Generator Coroutines

This notebook develops the classic generator-coroutine pattern in which a decorator
creates a generator and advances it to its first `yield`.

> **Scope:** These exercises teach generator-based coroutines and the `.send()`,
> `.throw()`, and `.close()` protocol. For new concurrent applications, prefer
> `async def` / `asyncio`; generator coroutines remain valuable for learning,
> streaming pipelines, state machines, and protocol design.

## Learning goals

By the end, you will be able to:

- write a production-quality priming decorator;
- preserve function metadata and support arbitrary arguments;
- validate coroutine construction and priming;
- recover from bad inputs without accidentally closing a coroutine;
- inject control signals with `.throw()`;
- guarantee cleanup with `.close()` and `finally`;
- compose filtering, transformation, fan-out, and aggregation stages;
- inspect and test coroutine state transitions;
- capture a generator's return value;
- build a reusable context-managed coroutine handle.

Each problem is followed by a complete solution and executable checks.

## 0. Core implementation used throughout the notebook

Best practices applied here:

1. `functools.wraps` preserves the wrapped function's name, documentation, and annotations.
2. `*args` and `**kwargs` support arbitrary coroutine factory signatures.
3. The wrapper verifies that the decorated callable really returns a generator.
4. Premature termination during priming is converted into a clear `RuntimeError`.
5. Type hints document the yielded, sent, and returned value types.

In [29]:
from __future__ import annotations

import inspect
import math
from collections import deque
from collections.abc import Callable, Generator, Iterable
from contextlib import AbstractContextManager
from dataclasses import dataclass
from functools import wraps
from numbers import Real
from typing import Any, Generic, ParamSpec, TypeVar, cast

P = ParamSpec("P")
YieldT = TypeVar("YieldT")
SendT = TypeVar("SendT")
ReturnT = TypeVar("ReturnT")


def primed(
    gen_fn: Callable[P, Generator[YieldT, SendT, ReturnT]],
) -> Callable[P, Generator[YieldT, SendT, ReturnT]]:
    """Create and advance a generator coroutine to its first yield."""

    @wraps(gen_fn)
    def wrapper(*args: P.args, **kwargs: P.kwargs) -> Generator[YieldT, SendT, ReturnT]:
        candidate = gen_fn(*args, **kwargs)

        if not inspect.isgenerator(candidate):
            raise TypeError(
                f"{gen_fn.__qualname__} must return a generator; "
                f"got {type(candidate).__name__}"
            )

        try:
            next(candidate)
        except StopIteration as exc:
            raise RuntimeError(
                f"{gen_fn.__qualname__} terminated before reaching its first yield"
            ) from exc

        return cast(Generator[YieldT, SendT, ReturnT], candidate)

    return wrapper

In [30]:
def generator_state(gen: Generator[Any, Any, Any]) -> str:
    """Small helper used by several exercises."""
    return inspect.getgeneratorstate(gen)


assert primed.__name__ == "primed"

## Warm-up example: echo with history

Unlike a plain printing coroutine, this version also stores received values so that
its behavior is easy to test.

In [31]:
@primed
def echo(history: list[str], prefix: str = "") -> Generator[None, str, None]:
    """Append every received message to history and print it."""
    while True:
        message = yield
        rendered = f"{prefix}{message}"
        history.append(rendered)
        print(rendered)


history: list[str] = []
ec = echo(history, prefix="received: ")

assert generator_state(ec) == "GEN_SUSPENDED"
ec.send("hello")
ec.send("world")
assert history == ["received: hello", "received: world"]

ec.close()
assert generator_state(ec) == "GEN_CLOSED"

received: hello
received: world


# Problem 1 — Preserve metadata and support arbitrary arguments

Implement a priming decorator that:

- accepts positional and keyword arguments;
- preserves `__name__` and `__doc__`;
- returns a generator already suspended at its first `yield`.

### Solution
The shared `@primed` decorator already meets these requirements. The following
coroutine verifies all three properties.

In [32]:
@primed
def affine_transform(
    scale: float,
    *,
    offset: float = 0.0,
) -> Generator[float | None, float, None]:
    """Return scale * value + offset for every sent value."""
    result: float | None = None
    while True:
        value = yield result
        result = scale * value + offset


assert affine_transform.__name__ == "affine_transform"
assert affine_transform.__doc__.startswith("Return scale")

transform = affine_transform(2.5, offset=1.0)
assert generator_state(transform) == "GEN_SUSPENDED"
assert transform.send(4.0) == 11.0
assert transform.send(-2.0) == -4.0
transform.close()

# Problem 2 — Reject invalid decorated callables clearly

A decorator can be applied incorrectly. Handle both cases:

1. the wrapped function returns a non-generator object;
2. the wrapped function creates a generator that returns before its first `yield`.

The caller should receive a useful exception instead of an obscure failure.

### Solution
The core decorator performs both checks. These tests demonstrate the behavior.

In [33]:
@primed
def not_a_generator() -> Any:
    return 42


def assert_raises(
    expected: type[BaseException],
    fn: Callable[[], Any],
    message_fragment: str,
) -> None:
    try:
        fn()
    except expected as exc:
        assert message_fragment in str(exc)
    else:
        raise AssertionError(f"Expected {expected.__name__}")


assert_raises(TypeError, not_a_generator, "must return a generator")

In [34]:
@primed
def terminates_during_priming() -> Generator[None, None, None]:
    if False:
        yield
    return


assert_raises(
    RuntimeError,
    terminates_during_priming,
    "terminated before reaching its first yield",
)

# Problem 3 — Build a resilient power coroutine

Create `power_up(exponent)` with these rules:

- numeric inputs produce `value ** exponent`;
- strings and other invalid values return `None`;
- booleans are rejected even though `bool` is a subclass of `int`;
- one invalid input must **not** close the coroutine;
- non-finite results (`inf` and `nan`) are rejected.

### Solution
Validate before calculation and catch only expected arithmetic errors.
Avoid a broad `except Exception`, which can hide programming bugs.

In [35]:
@primed
def power_up(exponent: Real) -> Generator[float | None, Real, None]:
    result: float | None = None

    while True:
        value = yield result

        if isinstance(value, bool) or not isinstance(value, Real):
            result = None
            continue

        try:
            candidate = math.pow(value, exponent)
        except (TypeError, ValueError, OverflowError):
            result = None
        else:
            result = candidate if math.isfinite(candidate) else None


squares = power_up(2)

assert squares.send(3) == 9.0
assert squares.send("abc") is None
assert generator_state(squares) == "GEN_SUSPENDED"
assert squares.send(True) is None
assert squares.send(1e200) is None
assert squares.send(-4) == 16.0

squares.close()

# Problem 4 — Online running statistics

Write a coroutine that receives one number at a time and returns a snapshot containing:

- count;
- sum;
- arithmetic mean;
- population variance;
- minimum;
- maximum.

Requirements:

- use an online update rather than storing all observations;
- reject non-real values and booleans without closing;
- return the previous valid snapshot after an invalid input.

### Solution
Welford's algorithm updates mean and variance in constant memory.

In [36]:
@dataclass(frozen=True)
class StatsSnapshot:
    count: int
    total: float
    mean: float
    variance: float
    minimum: float
    maximum: float


@primed
def running_statistics() -> Generator[StatsSnapshot | None, Real, None]:
    count = 0
    total = 0.0
    mean = 0.0
    m2 = 0.0
    minimum = math.inf
    maximum = -math.inf
    snapshot: StatsSnapshot | None = None

    while True:
        value = yield snapshot

        if isinstance(value, bool) or not isinstance(value, Real):
            continue

        x = float(value)
        count += 1
        total += x

        delta = x - mean
        mean += delta / count
        delta2 = x - mean
        m2 += delta * delta2

        minimum = min(minimum, x)
        maximum = max(maximum, x)

        snapshot = StatsSnapshot(
            count=count,
            total=total,
            mean=mean,
            variance=m2 / count,
            minimum=minimum,
            maximum=maximum,
        )

In [37]:
stats = running_statistics()

s1 = stats.send(2)
s2 = stats.send(4)
s3 = stats.send(8)

assert s1 == StatsSnapshot(1, 2.0, 2.0, 0.0, 2.0, 2.0)
assert s2 is not None and s2.mean == 3.0
assert s3 is not None and math.isclose(s3.mean, 14 / 3)
assert s3 is not None and math.isclose(s3.variance, 56 / 9)

same_snapshot = stats.send("bad input")
assert same_snapshot == s3
assert generator_state(stats) == "GEN_SUSPENDED"

stats.close()

# Problem 5 — Use `.throw()` for control signals

Design an accumulator that supports normal numeric input plus two injected commands:

- `ResetTotal`: reset the total to zero;
- `ScaleTotal(factor)`: multiply the total by a factor.

The commands must be delivered with `generator.throw(...)`, not with `.send()`.

### Solution
Exceptions thrown into a generator are raised at the suspended `yield`. Catch only
the intended protocol exceptions.

In [38]:
class ResetTotal(Exception):
    """Control signal requesting that the accumulator reset to zero."""


class ScaleTotal(Exception):
    """Control signal requesting that the accumulator scale its total."""

    def __init__(self, factor: float) -> None:
        super().__init__(factor)
        self.factor = factor


@primed
def controllable_accumulator() -> Generator[float, Real, None]:
    total = 0.0

    while True:
        try:
            value = yield total
        except ResetTotal:
            total = 0.0
        except ScaleTotal as command:
            total *= command.factor
        else:
            if isinstance(value, bool) or not isinstance(value, Real):
                continue
            total += float(value)

In [39]:
acc = controllable_accumulator()

assert acc.send(10) == 10.0
assert acc.send(2.5) == 12.5
assert acc.throw(ScaleTotal(2)) == 25.0
assert acc.throw(ResetTotal) == 0.0
assert acc.send(7) == 7.0

acc.close()

# Problem 6 — Guarantee cleanup on `.close()`

Create a buffered sink that appends every received item to a buffer and records a
cleanup event when closed.

Requirements:

- cleanup must happen exactly once;
- closing an already closed generator must be harmless;
- the coroutine must be closed even if an exception occurs in the caller.

### Solution
Put cleanup logic in `finally`, and close from a caller-side `try/finally`.

In [40]:
@primed
def buffered_sink(
    buffer: list[Any],
    lifecycle: list[str],
) -> Generator[None, Any, None]:
    lifecycle.append("opened")
    try:
        while True:
            item = yield
            buffer.append(item)
    finally:
        lifecycle.append("closed")


buffer: list[Any] = []
lifecycle: list[str] = []
sink = buffered_sink(buffer, lifecycle)

try:
    sink.send({"id": 1})
    sink.send({"id": 2})
finally:
    sink.close()

sink.close()  # harmless second close

assert buffer == [{"id": 1}, {"id": 2}]
assert lifecycle == ["opened", "closed"]
assert generator_state(sink) == "GEN_CLOSED"

# Problem 7 — Compose a filter → transform → sink pipeline

Build three reusable coroutine stages:

- `collect(target_list)` appends received values;
- `map_stage(function, target)` transforms each value;
- `filter_stage(predicate, target)` forwards only matching values.

Closing the first stage must close every downstream stage.

### Solution
Each forwarding stage closes its target in `finally`.

In [41]:
@primed
def collect(target_list: list[Any]) -> Generator[None, Any, None]:
    while True:
        target_list.append((yield))


@primed
def map_stage(
    function: Callable[[Any], Any],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            item = yield
            target.send(function(item))
    finally:
        target.close()


@primed
def filter_stage(
    predicate: Callable[[Any], bool],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        target.close()

In [42]:
pipeline_output: list[int] = []

pipeline = filter_stage(
    lambda n: n % 2 == 0,
    map_stage(lambda n: n * n, collect(pipeline_output)),
)

for number in range(10):
    pipeline.send(number)

pipeline.close()

assert pipeline_output == [0, 4, 16, 36, 64]
assert generator_state(pipeline) == "GEN_CLOSED"

# Problem 8 — Fan out to multiple targets and remove failed targets

Implement a broadcaster that forwards each item to all active targets.

Advanced requirements:

- one closed target must not prevent delivery to healthy targets;
- failed targets are removed from future broadcasts;
- closing the broadcaster closes all still-active targets;
- the returned value is the number of active targets after each broadcast.

### Solution
Catch `StopIteration` only around each target's `.send()`.

In [43]:
@primed
def broadcaster(
    *targets: Generator[Any, Any, Any],
) -> Generator[int, Any, None]:
    active = list(targets)
    active_count = len(active)

    try:
        while True:
            item = yield active_count
            survivors: list[Generator[Any, Any, Any]] = []

            for target in active:
                try:
                    target.send(item)
                except StopIteration:
                    pass
                else:
                    survivors.append(target)

            active = survivors
            active_count = len(active)
    finally:
        for target in active:
            target.close()

In [44]:
left: list[str] = []
right: list[str] = []

left_sink = collect(left)
right_sink = collect(right)
fanout = broadcaster(left_sink, right_sink)

assert fanout.send("A") == 2
right_sink.close()
assert fanout.send("B") == 1
assert fanout.send("C") == 1

fanout.close()

assert left == ["A", "B", "C"]
assert right == ["A"]
assert generator_state(left_sink) == "GEN_CLOSED"

# Problem 9 — Sliding-window moving average

Create `moving_average(window_size)`:

- `window_size` must be a positive integer;
- keep only the most recent values;
- return the current average after each valid input;
- reject invalid inputs without changing the window;
- use `collections.deque(maxlen=...)`.

### Solution
The deque enforces bounded memory automatically.

In [45]:
@primed
def moving_average(window_size: int) -> Generator[float | None, Real, None]:
    if isinstance(window_size, bool) or not isinstance(window_size, int) or window_size <= 0:
        raise ValueError("window_size must be a positive integer")

    values: deque[float] = deque(maxlen=window_size)
    average: float | None = None

    while True:
        value = yield average

        if isinstance(value, bool) or not isinstance(value, Real):
            continue

        values.append(float(value))
        average = sum(values) / len(values)

In [46]:
ma = moving_average(3)

assert ma.send(3) == 3.0
assert ma.send(6) == 4.5
assert ma.send(9) == 6.0
assert ma.send(12) == 9.0
assert ma.send("skip") == 9.0
assert generator_state(ma) == "GEN_SUSPENDED"

ma.close()

assert_raises(ValueError, lambda: moving_average(0), "positive integer")

# Problem 10 — Enforce a priming handshake

Sometimes the first yielded value is part of a protocol. Build a decorator factory
that primes a coroutine and verifies that its initial yielded value equals an
expected handshake token.

Requirements:

- preserve metadata;
- close the generator before raising on handshake mismatch;
- support arbitrary arguments.

### Solution
Use the value returned by `next(generator)` as the handshake.

In [47]:
HandshakeT = TypeVar("HandshakeT")


def primed_with_handshake(
    expected: HandshakeT,
) -> Callable[
    [Callable[P, Generator[HandshakeT, SendT, ReturnT]]],
    Callable[P, Generator[HandshakeT, SendT, ReturnT]],
]:
    def decorate(
        gen_fn: Callable[P, Generator[HandshakeT, SendT, ReturnT]],
    ) -> Callable[P, Generator[HandshakeT, SendT, ReturnT]]:
        @wraps(gen_fn)
        def wrapper(
            *args: P.args,
            **kwargs: P.kwargs,
        ) -> Generator[HandshakeT, SendT, ReturnT]:
            gen = gen_fn(*args, **kwargs)

            if not inspect.isgenerator(gen):
                raise TypeError(f"{gen_fn.__qualname__} must return a generator")

            try:
                actual = next(gen)
            except StopIteration as exc:
                raise RuntimeError("Coroutine terminated before handshake") from exc

            if actual != expected:
                gen.close()
                raise RuntimeError(
                    f"Handshake mismatch: expected {expected!r}, got {actual!r}"
                )

            return gen

        return wrapper

    return decorate

In [48]:
READY = object()


@primed_with_handshake(READY)
def protocol_endpoint(log: list[str]) -> Generator[object | str, str, None]:
    message = yield READY
    while True:
        log.append(message)
        message = yield f"ACK:{message}"


protocol_log: list[str] = []
endpoint = protocol_endpoint(protocol_log)

assert endpoint.send("one") == "ACK:one"
assert endpoint.send("two") == "ACK:two"
assert protocol_log == ["one", "two"]

endpoint.close()

In [49]:
@primed_with_handshake("READY")
def bad_endpoint() -> Generator[str, str, None]:
    yield "NOT_READY"
    while True:
        yield "unused"


assert_raises(RuntimeError, bad_endpoint, "Handshake mismatch")

# Problem 11 — Capture a generator's return value

Calling `.close()` raises `GeneratorExit` inside the generator and does not expose a
normal return value. Design a coroutine that terminates when sent a unique sentinel
and returns the collected values through `StopIteration.value`.

### Solution
Send a private sentinel and catch `StopIteration` in the caller.

In [50]:
STOP = object()


@primed
def collecting_session() -> Generator[int, int | object, tuple[int, ...]]:
    items: list[int] = []

    while True:
        item = yield len(items)
        if item is STOP:
            return tuple(items)
        if isinstance(item, bool) or not isinstance(item, int):
            continue
        items.append(item)


session = collecting_session()

assert session.send(10) == 1
assert session.send(20) == 2
assert session.send("ignored") == 2

try:
    session.send(STOP)
except StopIteration as exc:
    collected = exc.value
else:
    raise AssertionError("The coroutine should have terminated")

assert collected == (10, 20)
assert generator_state(session) == "GEN_CLOSED"

# Problem 12 — Build a context-managed coroutine handle

Create a wrapper class that:

- owns a generator coroutine;
- exposes `.send()` and `.throw()`;
- reports the current generator state;
- closes automatically at the end of a `with` block;
- rejects operations after closure with a clear error.

### Solution
`AbstractContextManager` provides the context-manager protocol while the wrapped
generator retains the coroutine protocol.

In [51]:
OutT = TypeVar("OutT")
InT = TypeVar("InT")
RetT = TypeVar("RetT")


class CoroutineHandle(
    AbstractContextManager["CoroutineHandle[OutT, InT, RetT]"],
    Generic[OutT, InT, RetT],
):
    def __init__(self, generator: Generator[OutT, InT, RetT]) -> None:
        if not inspect.isgenerator(generator):
            raise TypeError("generator must be a generator object")
        self._generator = generator

    @property
    def state(self) -> str:
        return inspect.getgeneratorstate(self._generator)

    @property
    def closed(self) -> bool:
        return self.state == "GEN_CLOSED"

    def _require_open(self) -> None:
        if self.closed:
            raise RuntimeError("coroutine handle is closed")

    def send(self, value: InT) -> OutT:
        self._require_open()
        return self._generator.send(value)

    def throw(self, error: BaseException | type[BaseException]) -> OutT:
        self._require_open()
        return self._generator.throw(error)

    def close(self) -> None:
        self._generator.close()

    def __exit__(self, exc_type, exc, traceback) -> None:
        self.close()
        return None

In [52]:
with CoroutineHandle(controllable_accumulator()) as handle:
    assert handle.state == "GEN_SUSPENDED"
    assert handle.send(5) == 5.0
    assert handle.throw(ScaleTotal(3)) == 15.0

assert handle.closed
assert_raises(RuntimeError, lambda: handle.send(1), "closed")

# Problem 13 — Debug a subtly broken priming decorator

The decorator below has several defects:

```python
def broken(gen_fn):
    def wrapper(args, kwargs):
        gen = gen_fn(args, kwargs)
        gen.send(1)
        return gen
    return wrapper
```

Find and fix at least four issues.

### Defects

1. It does not accept arbitrary positional and keyword arguments correctly.
2. It calls the wrapped function with two container objects instead of forwarding arguments.
3. Initial priming must use `next(gen)` or `gen.send(None)`, never `gen.send(1)`.
4. It does not preserve metadata with `functools.wraps`.
5. It does not verify that the result is a generator.
6. It gives poor diagnostics if the generator stops during priming.

### Solution
The notebook's shared `@primed` decorator is the corrected implementation.
The next cell provides a compact regression test.

In [53]:
@primed
def configurable_counter(
    start: int,
    *,
    step: int = 1,
) -> Generator[int, None, None]:
    current = start
    while True:
        yield current
        current += step


counter = configurable_counter(10, step=3)

# Because the first value (10) was consumed during priming, subsequent next()
# calls produce 13, 16, and 19.
assert next(counter) == 13
assert next(counter) == 16
assert next(counter) == 19
assert configurable_counter.__name__ == "configurable_counter"

counter.close()

# Problem 14 — Capstone: resilient event-processing pipeline

Build a pipeline for dictionaries representing application events.

Input examples:

```python
{"type": "purchase", "user": "u1", "amount": 12.50}
{"type": "login", "user": "u2"}
{"type": "purchase", "user": "u1", "amount": "bad"}
```

Requirements:

1. Validate that each event is a dictionary.
2. Keep only `"purchase"` events.
3. Parse `amount` as a finite non-negative float.
4. Route malformed events to an error sink without stopping the pipeline.
5. Aggregate total purchase amount per user.
6. Expose an immutable snapshot after every valid purchase.
7. Closing the entry stage must close all downstream stages.

### Solution
The design separates validation, routing, and aggregation into focused stages.

In [54]:
@dataclass(frozen=True)
class PipelineError:
    event: Any
    reason: str


@primed
def error_sink(errors: list[PipelineError]) -> Generator[None, PipelineError, None]:
    while True:
        errors.append((yield))


@primed
def purchase_aggregator(
    snapshots: list[dict[str, float]],
) -> Generator[None, dict[str, Any], None]:
    totals: dict[str, float] = {}

    while True:
        event = yield
        user = event["user"]
        amount = event["amount"]

        totals[user] = totals.get(user, 0.0) + amount
        snapshots.append(dict(totals))


@primed
def purchase_parser(
    target: Generator[Any, dict[str, Any], Any],
    errors: Generator[Any, PipelineError, Any],
) -> Generator[None, dict[str, Any], None]:
    try:
        while True:
            event = yield

            user = event.get("user")
            raw_amount = event.get("amount")

            if not isinstance(user, str) or not user:
                errors.send(PipelineError(event, "missing or invalid user"))
                continue

            if isinstance(raw_amount, bool):
                errors.send(PipelineError(event, "boolean amount is not allowed"))
                continue

            try:
                amount = float(raw_amount)
            except (TypeError, ValueError):
                errors.send(PipelineError(event, "amount is not numeric"))
                continue

            if not math.isfinite(amount) or amount < 0:
                errors.send(PipelineError(event, "amount must be finite and non-negative"))
                continue

            target.send({"type": "purchase", "user": user, "amount": amount})
    finally:
        target.close()
        errors.close()


@primed
def event_router(
    purchases: Generator[Any, dict[str, Any], Any],
    errors: Generator[Any, PipelineError, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            event = yield

            if not isinstance(event, dict):
                errors.send(PipelineError(event, "event must be a dictionary"))
                continue

            if event.get("type") != "purchase":
                continue

            purchases.send(event)
    finally:
        purchases.close()
        # purchase_parser closes the shared error sink.

In [55]:
snapshots: list[dict[str, float]] = []
errors: list[PipelineError] = []

errors_stage = error_sink(errors)
aggregate_stage = purchase_aggregator(snapshots)
parse_stage = purchase_parser(aggregate_stage, errors_stage)
entry_stage = event_router(parse_stage, errors_stage)

events = [
    {"type": "purchase", "user": "u1", "amount": 12.50},
    {"type": "login", "user": "u2"},
    {"type": "purchase", "user": "u1", "amount": "7.5"},
    {"type": "purchase", "user": "u2", "amount": 3},
    {"type": "purchase", "user": "u1", "amount": "bad"},
    {"type": "purchase", "user": "", "amount": 2},
    {"type": "purchase", "user": "u3", "amount": float("inf")},
    ["not", "a", "dict"],
]

for event in events:
    entry_stage.send(event)

entry_stage.close()

assert snapshots == [
    {"u1": 12.5},
    {"u1": 20.0},
    {"u1": 20.0, "u2": 3.0},
]

assert [error.reason for error in errors] == [
    "amount is not numeric",
    "missing or invalid user",
    "amount must be finite and non-negative",
    "event must be a dictionary",
]

assert generator_state(entry_stage) == "GEN_CLOSED"
assert generator_state(parse_stage) == "GEN_CLOSED"
assert generator_state(aggregate_stage) == "GEN_CLOSED"
assert generator_state(errors_stage) == "GEN_CLOSED"

# Additional challenge problems

These are intentionally left as extension exercises. Suggested solution strategies
are included so that you can continue experimenting.

## Challenge A — Backpressure simulation

Add a target that accepts at most `N` items and then raises a custom `BufferFull`
exception. Update the broadcaster so it can either drop the item, retry later, or
remove the target according to a configurable policy.

**Suggested approach:** represent the policy with an `Enum`; catch only
`BufferFull`; keep retry state outside the target.

## Challenge B — Timestamped rate monitor

Write a coroutine that receives `(timestamp, value)` pairs and returns the number
of events observed in the trailing 60 seconds.

**Suggested approach:** store timestamps in a `deque`; remove entries older than
`timestamp - 60`; reject non-monotonic timestamps explicitly.

## Challenge C — Hierarchical routing

Route events by `"type"` to different coroutine pipelines. Unknown types should go
to a dead-letter sink.

**Suggested approach:** maintain `dict[str, Generator]`; look up the target for each
event; close every registered target in `finally`.

## Challenge D — Coroutine test harness

Build a helper that accepts a coroutine factory, a sequence of sent values, and a
sequence of expected yielded results. It should also verify generator state before
and after closure.

**Suggested approach:** use `inspect.getgeneratorstate`, `zip(..., strict=True)`,
and a caller-side `try/finally`.

# Best-practice checklist

- Prime with `next(gen)` or `gen.send(None)` only.
- Preserve metadata with `@wraps`.
- Forward arguments with `*args, **kwargs`.
- Validate decorated callables and report premature termination clearly.
- Catch narrow, expected exceptions inside the coroutine.
- Use dedicated exception types for `.throw()` control messages.
- Put resource cleanup and downstream closure in `finally`.
- Test `GEN_SUSPENDED` and `GEN_CLOSED` states when lifecycle matters.
- Do not silently swallow programmer errors.
- Use finite-memory algorithms for long-running streams.
- Prefer immutable snapshots when exposing internal state.
- Use `async def` and `asyncio` for modern asynchronous I/O and concurrency.

# Final verification

Reaching this cell after **Run All** means every assertion in the notebook passed.

In [56]:
print("All advanced primed-coroutine examples and solutions passed.")

All advanced primed-coroutine examples and solutions passed.
